# Elliptic Bitcoin Transactions: Exploratory Data Analysis

Notebook 1 of 7 in a chained pipeline: EDA, features, GraphSAGE tuning, GraphSAGE training, evaluation, random-split contrast, then the serving export. This notebook studies the raw data and motivates the choices made in notebook 2. Nothing it computes is consumed downstream.

The Elliptic Data Set is a Bitcoin transaction graph: nodes are transactions, a directed edge is a flow of bitcoins from one transaction to the next, and each node carries 166 anonymised features and a label of illicit, licit or unknown.

## Setup

Imports, plotting style and helpers. `find_file` searches the Kaggle inputs so the notebook does not depend on how Kaggle mounts the dataset, `save` writes every figure to `figures/`, and `elapsed` reports runtime. `SMOKE` switches to a quick check run on five time steps; it is off in this version.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from cycler import cycler
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

SMOKE = False
SEED = 42
TRAIN_LAST = 34
SMOKE_STEPS = [1, 2, 3, 35, 36]
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
CLASS_COLORS = {"licit": "#2a78d6", "illicit": "#eb6834", "unknown": "#898781"}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "text.color": INK, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2, "legend.frameon": False,
    "font.family": "sans-serif", "axes.prop_cycle": cycler(color=SERIES), "axes.titlesize": 11,
})
LOCAL = [f"local_{i}" for i in range(1, 94)]
AGG = [f"agg_{i}" for i in range(1, 73)]
RAW = LOCAL + AGG
NAMES = {1: "illicit", 0: "licit", -1: "unknown"}
START = time.time()


def find_file(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def elapsed():
    return f"elapsed {(time.time() - START) / 60:.1f} minutes"


print(f"smoke={SMOKE}")
print(f"pandas {pd.__version__}, numpy {np.__version__}, networkx {nx.__version__}")

**Conclusion.** The smoke switch is off, so this run reads the full dataset. Kaggle's image provides pandas 2.3.3, NumPy 2.1.3 and NetworkX 3.6.1, so nothing needs installing for this notebook or for notebook 2.

## Load the three files

Aim: confirm the node and edge counts behind the 203K and 234K figures, name the anonymised columns (`local_1` to `local_93`, `agg_1` to `agg_72`, plus the time step), and check the edge list for self-loops, duplicates and dangling endpoints before any analysis. Features are stored as float32, the precision every later notebook uses.

In [ ]:
features = pd.read_csv(find_file("elliptic_txs_features.csv"), header=None)
features.columns = ["txId", "time_step"] + RAW
features[RAW] = features[RAW].astype(np.float32)
classes = pd.read_csv(find_file("elliptic_txs_classes.csv"))
classes["class"] = classes["class"].astype(str)
edges = pd.read_csv(find_file("elliptic_txs_edgelist.csv"))
features = features.merge(classes, on="txId", how="left")
features["label"] = features["class"].map({"1": 1, "2": 0, "unknown": -1}).astype(np.int8)
if SMOKE:
    features = features[features.time_step.isin(SMOKE_STEPS)].reset_index(drop=True)
    keep = set(features.txId)
    edges = edges[edges.txId1.isin(keep) & edges.txId2.isin(keep)].reset_index(drop=True)
ids = set(features.txId)
print(f"nodes: {len(features):,}")
print(f"edges: {len(edges):,}")
print(f"feature columns: {len(RAW)} plus time step ({len(LOCAL)} local, {len(AGG)} aggregated)")
print(f"time steps: {features.time_step.min()} to {features.time_step.max()}")
print(f"duplicate txIds: {features.txId.duplicated().sum()}")
print(f"nodes without a class row: {features['class'].isna().sum()}")
print(f"self-loops: {(edges.txId1 == edges.txId2).sum()}")
print(f"duplicate edges: {edges.duplicated().sum()}")
print(f"edges with an endpoint outside the node table: {(~edges.txId1.isin(ids) | ~edges.txId2.isin(ids)).sum()}")
print(f"feature table memory: {features.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(elapsed())

**Conclusion.** The graph has 203,769 transactions and 234,355 directed edges across time steps 1 to 49, matching the dataset description, with 165 features per node plus the time step (93 local, 72 aggregated). The data is clean: no duplicate transactions, no missing class rows, no self-loops, no duplicate edges and no edge pointing outside the node table. In float32 the feature table takes 149 MB.

## Label balance

Aim: measure how rare illicit transactions are, both as a share of every node and as a share of labelled nodes, overall and in the training (steps 1 to 34) and test (steps 35 to 49) periods. The resume entry quotes "2.3% illicit"; this cell gives the exact figure. With a rare positive class, accuracy is meaningless, so the pipeline reports AUC-ROC, PR-AUC and illicit-class F1.

In [ ]:
counts = features["label"].map(NAMES).value_counts()
print(counts.to_string())
total = len(features)
labelled = int((features.label >= 0).sum())
illicit = int((features.label == 1).sum())
print(f"labelled nodes: {labelled:,} ({labelled / total:.2%} of all)")
print(f"illicit share of all nodes: {illicit / total:.4%}")
print(f"illicit share of labelled nodes: {illicit / labelled:.4%}")
for name, mask in [("train steps 1-34", features.time_step <= TRAIN_LAST), ("test steps 35-49", features.time_step > TRAIN_LAST)]:
    part = features[mask]
    lab = int((part.label >= 0).sum())
    ill = int((part.label == 1).sum())
    print(f"{name}: nodes {len(part):,}, labelled {lab:,}, illicit {ill:,}, licit {lab - ill:,}, illicit of all {ill / len(part):.4%}, illicit of labelled {ill / lab:.4%}")

**Conclusion.** 4,545 transactions are illicit, 42,019 licit and 157,205 unlabelled. Illicit transactions are 2.23% of all nodes and 9.76% of labelled ones, so the resume's "2.3% illicit" corresponds to the share of all nodes, which rounds to 2.2%. The two periods differ: illicit transactions are 11.58% of labelled nodes in steps 1 to 34 (3,462 of 29,894) but only 6.50% in steps 35 to 49 (1,083 of 16,670), so a threshold tuned on the training period meets a rarer positive class at test time.

## Labels over time

Aim: see how the illicit count and the illicit rate move across the 49 time steps, and where the train and test periods split. The literature reports a dark-market shutdown around step 43 after which models stop finding illicit activity; if it is real, the illicit count should collapse there.

In [ ]:
per_step = features.groupby("time_step").agg(
    nodes=("txId", "size"),
    labelled=("label", lambda s: int((s >= 0).sum())),
    illicit=("label", lambda s: int((s == 1).sum())),
)
per_step["licit"] = per_step["labelled"] - per_step["illicit"]
per_step["illicit_rate"] = per_step["illicit"] / per_step["labelled"]
print(per_step.to_string(float_format=lambda v: f"{v:.4f}"))
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(per_step.index, per_step["illicit"], color=CLASS_COLORS["illicit"], width=0.75)
ax.axvline(TRAIN_LAST + 0.5, color=MUTED, linestyle="--", linewidth=1)
ax.text(TRAIN_LAST + 0.8, per_step["illicit"].max() * 0.95, "test period", color=INK_2, va="top")
ax.set(xlabel="time step", ylabel="illicit transactions", title="Illicit transactions per time step")
save(fig, "illicit_by_step")
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(per_step.index, per_step["illicit_rate"], color=SERIES[0], marker="o", markersize=5)
ax.axvline(TRAIN_LAST + 0.5, color=MUTED, linestyle="--", linewidth=1)
ax.set(xlabel="time step", ylabel="illicit share of labelled", title="Illicit rate among labelled transactions per time step")
save(fig, "illicit_rate_by_step")

**Conclusion.** The illicit rate swings widely from step to step, from 0.4% at step 5 to 36% at step 13, so neighbouring steps are not interchangeable. The shutdown is clear in the test period: step 42 has 239 illicit transactions (11.1% of labelled), step 43 has 24 (1.8%), and steps 45 and 46 have only 5 and 2. A model trained on steps 1 to 34 meets a regime it never saw from step 43 on, so the evaluation reports steps 35 to 42 and 43 to 49 separately.

## Graph structure

Aim: verify the dataset's claims that no edge crosses time steps and that each step is one connected component, then describe the in-degree and out-degree distributions. No cross-step edges is what makes time-step folds leak-free: a fold's nodes and their whole neighbourhoods drop out of training together.

In [ ]:
step_of = features.set_index("txId")["time_step"]
crossing = (edges.txId1.map(step_of) != edges.txId2.map(step_of)).sum()
print(f"edges crossing time steps: {crossing}")
components = {}
for step, nodes in features.groupby("time_step")["txId"]:
    graph = nx.Graph()
    graph.add_nodes_from(nodes)
    part = edges[edges.txId1.isin(set(nodes))]
    graph.add_edges_from(zip(part.txId1, part.txId2))
    components[step] = nx.number_connected_components(graph)
print(f"steps that form one weakly connected component: {sum(v == 1 for v in components.values())} of {len(components)}")
print(f"most components in one step: {max(components.values())}")
in_deg = edges.txId2.value_counts().reindex(features.txId, fill_value=0)
out_deg = edges.txId1.value_counts().reindex(features.txId, fill_value=0)
for name, deg in [("in", in_deg), ("out", out_deg)]:
    print(f"{name}-degree: mean {deg.mean():.3f}, median {deg.median():.0f}, max {deg.max()}, share zero {(deg == 0).mean():.2%}")
fig, ax = plt.subplots(figsize=(6.4, 4))
for name, deg, color in [("in-degree", in_deg, SERIES[0]), ("out-degree", out_deg, SERIES[1])]:
    values = np.sort(deg.to_numpy())
    values = values[values > 0]
    ccdf = 1.0 - np.arange(len(values)) / len(values)
    ax.loglog(values, ccdf, color=color, label=name)
ax.set(xlabel="degree", ylabel="share of nodes with at least this degree", title="Degree distribution (complementary CDF)")
ax.legend()
save(fig, "degree_ccdf")
print(elapsed())

**Conclusion.** No edge crosses time steps and each of the 49 steps is a single weakly connected component, exactly as the dataset documents, which makes folds of whole time steps leak-free. Degrees are heavy-tailed: the median in-degree and out-degree are both 1, but the largest are 284 and 472. 27% of transactions have no inputs in the graph and 18% have no outputs, so neighbourhood aggregates must handle empty neighbourhoods.

## Degree by class

Aim: check whether illicit transactions sit in structurally different places, more or fewer inputs and outputs, which is the first hint that graph features can add signal beyond the node's own attributes.

In [ ]:
deg = pd.DataFrame({"in_degree": in_deg.to_numpy(), "out_degree": out_deg.to_numpy(), "class": features.label.map(NAMES).to_numpy()})
print(deg.groupby("class").agg(["mean", "median", "max"]).round(3).to_string())

**Conclusion.** Illicit transactions have fewer outputs than licit ones (mean out-degree 0.74 against 1.19, never more than 3, while licit transactions reach 472) and fewer inputs (mean in-degree 1.27 against 1.91). Structure alone separates the classes somewhat, which supports adding graph features to the node's own attributes.

## Feature summary

Aim: check the scale and completeness of the anonymised features. The features are reported to be standardised already; missing values or constant columns would need handling before modelling.

In [ ]:
desc = features[RAW].describe().T
print(f"missing values: {int(features[RAW].isna().sum().sum())}")
print(f"constant columns: {int((desc['std'] == 0).sum())}")
print(f"local features: mean of column means {desc.loc[LOCAL, 'mean'].mean():.4f}, mean of column stds {desc.loc[LOCAL, 'std'].mean():.4f}")
print(f"aggregated features: mean of column means {desc.loc[AGG, 'mean'].mean():.4f}, mean of column stds {desc.loc[AGG, 'std'].mean():.4f}")
print(desc.loc[LOCAL[:10], ["mean", "std", "min", "max"]].round(4).to_string())

**Conclusion.** There are no missing values and no constant columns. Every feature is already standardised over the whole dataset (column means 0, standard deviations 1) but has a long right tail (local_5 reaches 260 standard deviations). Tree models are unaffected by scale; GraphSAGE inputs are restandardised on training-period statistics only, so the test period never informs the scaling.

## Drift between train and test

Aim: measure how far each local feature's distribution moves from the training period to the test period with the two-sample Kolmogorov-Smirnov statistic. Strong drift explains why random splits flatter models and why the temporal split is the honest protocol.

In [ ]:
train = features[features.time_step <= TRAIN_LAST]
test = features[features.time_step > TRAIN_LAST]
ks = pd.Series({c: ks_2samp(train[c], test[c]).statistic for c in LOCAL}).sort_values(ascending=False)
print(ks.head(15).round(4).to_string())
print(f"local features with KS above 0.1: {int((ks > 0.1).sum())} of {len(ks)}")
top = ks.head(15)[::-1]
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.barh(top.index, top.to_numpy(), color=SERIES[0], height=0.7)
for y, value in enumerate(top.to_numpy()):
    ax.text(value, y, f" {value:.2f}", va="center", color=INK_2, fontsize=8)
ax.set(xlabel="KS statistic, steps 1-34 against 35-49", title="Local features that drift most between train and test")
save(fig, "drift_ks")

**Conclusion.** Distribution shift between the periods is substantial: 31 of 93 local features have a KS statistic above 0.1, led by local_2 (0.60) and local_3 (0.47). A model that leans on drifting features degrades on the test period, which is also why a random split, mixing both periods, overstates performance.

## Univariate separation

Aim: rank raw features by how well each one alone separates illicit from licit transactions in the training period (distance of the single-feature AUC from 0.5). It previews which local attributes notebook 2 is likely to select for neighbourhood aggregation.

In [ ]:
lab = train[train.label >= 0]
auc = pd.Series({c: roc_auc_score(lab.label, lab[c]) for c in RAW})
separation = (auc - 0.5).abs().sort_values(ascending=False)
print(pd.DataFrame({"auc": auc[separation.index[:15]], "separation": separation.head(15)}).round(4).to_string())
print(f"local features among the 34 most separating: {sum(c in LOCAL for c in separation.index[:34])}")
print(elapsed())

**Conclusion.** Single features already separate the classes strongly in the training period: local_53 alone has an AUC of 0.145 (0.855 with its sign flipped, so illicit transactions take lower values), and 26 of the 34 most separating features are local ones. Notebook 2 therefore draws the 34 attributes for neighbourhood aggregation from the 93 local features, chosen by a multivariate model rather than these univariate scores.

## Summary

What notebook 2 takes from this analysis:

- No edge crosses time steps, so features are computed per step, and folds of whole time steps keep every node's neighbourhood on one side of a split.
- The time step is not a model input, because test steps 35 to 49 lie outside the training range.
- Illicit transactions are 2.2% of all nodes and 9.8% of labelled ones, and rarer in the test period than in training, so ranking metrics and illicit-class F1 are the yardsticks.
- Empty neighbourhoods are common, so an aggregate is missing, not zero, when a node has no inputs or outputs.
- Local features carry most of the single-feature signal, so the 34 aggregated attributes come from them.
- The test period holds a regime change at step 43 that no validation inside steps 1 to 34 can anticipate.